In [25]:
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report

In [26]:
model  = joblib.load('heart_disease_model.pkl')
scaler = joblib.load('scaler.pkl')

In [27]:
features = [
    'Number of vessels fluro',
    'Sex',
    'Chest pain type',
    'Thallium',
    'Exercise angina',
    'Max HR',
    'BP',
    'Slope of ST',
    'Cholesterol',
    'FBS over 120',
    'ST depression'
]

print(f"Expecting {len(features)} features")

Expecting 11 features


In [28]:
# ── Load Test Set (no guessing, no manual entry) ──────────────
X_test        = pd.read_csv('test_set.csv')
actual_labels = pd.read_csv('test_labels.csv').values.ravel()

print(f"Test set loaded → {len(X_test)} patients")
print(f"Columns in test set: {X_test.columns.tolist()}")

Test set loaded → 54 patients
Columns in test set: ['BP', 'Cholesterol', 'Chest pain type', 'Sex', 'FBS over 120', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


In [29]:
X_test = X_test[features]

df_scaled = scaler.transform(X_test)

predictions = model.predict(df_scaled)

probabilities = model.predict_proba(df_scaled)

print("Predictions complete!")

Predictions complete!


In [30]:
# results

X_test['Prediction'] = ['Heart Disease' if p == 1 
                         else 'No Heart Disease' 
                         for p in predictions]

X_test['Confidence'] = [f"{max(p):.2%}" for p in probabilities]

X_test['Actual'] = ['Heart Disease' if a == 1 
                     else 'No Heart Disease' 
                     for a in actual_labels]

X_test['Correct'] = X_test['Prediction'] == X_test['Actual']

print("\n── Patient Predictions ──────────────────────────────")
print(X_test[['Prediction', 'Actual', 'Correct', 'Confidence']].to_string())


── Patient Predictions ──────────────────────────────
          Prediction            Actual  Correct Confidence
0   No Heart Disease  No Heart Disease     True     92.20%
1   No Heart Disease  No Heart Disease     True     84.10%
2   No Heart Disease  No Heart Disease     True     93.45%
3      Heart Disease     Heart Disease     True     98.70%
4   No Heart Disease  No Heart Disease     True     90.91%
5      Heart Disease  No Heart Disease    False     91.91%
6   No Heart Disease  No Heart Disease     True     92.44%
7      Heart Disease     Heart Disease     True     89.12%
8      Heart Disease     Heart Disease     True     54.41%
9      Heart Disease     Heart Disease     True     67.38%
10     Heart Disease  No Heart Disease    False     91.92%
11     Heart Disease     Heart Disease     True     68.73%
12  No Heart Disease  No Heart Disease     True     84.80%
13     Heart Disease  No Heart Disease    False     86.02%
14     Heart Disease     Heart Disease     True     79.06%
1

In [31]:
print("\n── Accuracy Assessment ──────────────────────────────")
print(f"Accuracy : {accuracy_score(actual_labels, predictions):.4f}")

print(classification_report(
    actual_labels,
    predictions,
    target_names=['No Heart Disease', 'Heart Disease']
))



── Accuracy Assessment ──────────────────────────────
Accuracy : 0.7778
                  precision    recall  f1-score   support

No Heart Disease       0.91      0.67      0.77        30
   Heart Disease       0.69      0.92      0.79        24

        accuracy                           0.78        54
       macro avg       0.80      0.79      0.78        54
    weighted avg       0.81      0.78      0.78        54



In [32]:
new_patient = pd.DataFrame([{
    'Number of vessels fluro' : 2,    # 0,1,2,3 blocked vessels
    'Sex'                     : 1,    # 1=Male 0=Female
    'Chest pain type'         : 3,    # 1,2,3,4
    'Thallium'                : 6,    # 3=normal 6=fixed 7=reversible
    'Exercise angina'         : 0,    # 1=Yes 0=No
    'Max HR'                  : 150,  # maximum heart rate
    'BP'                      : 130,  # resting blood pressure
    'Slope of ST'             : 2,    # 1,2,3
    'Cholesterol'             : 230,  # mg/dl
    'FBS over 120'            : 0,    # 1=Yes 0=No
    'ST depression'           : 1.5,  # ST depression value
}])

In [33]:
new_patient = new_patient[features]

new_scaled = scaler.transform(new_patient)

new_prediction  = model.predict(new_scaled)[0]
new_probability = model.predict_proba(new_scaled)[0]

print("\n── New Patient Result ───────────────────────────────")
print(f"Prediction  : {'Heart Disease' if new_prediction == 1 else 'No Heart Disease'}")
print(f"Confidence  : {max(new_probability):.2%}")
print(f"Breakdown   : No Disease = {new_probability[0]:.2%}  |  Disease = {new_probability[1]:.2%}")


── New Patient Result ───────────────────────────────
Prediction  : Heart Disease
Confidence  : 80.32%
Breakdown   : No Disease = 19.68%  |  Disease = 80.32%
